Import libraries

In [ ]:
import pandas as pd
import numpy as np

from linearmodels.panel import PanelOLS, RandomEffects
from scipy import stats
import statsmodels.api as sm

import warnings
warnings.filterwarnings("ignore")

import os

Caricamento Dati, Import & Esplorazione Dataset

In [3]:
df18 = pd.read_csv("18-19.csv")
df19 = pd.read_csv("19-20.csv")
df20 = pd.read_csv("20-21.csv")
df21 = pd.read_csv("21-22.csv")
df22 = pd.read_csv("22-23.csv")


Nome pulito 

In [ ]:
# File names and seasons

files = {
    "2018-19": "18-19.csv",
    "2019-20": "19-20.csv",
    "2020-21": "20-21.csv",
    "2021-22": "21-22.csv",
    "2022-23": "22-23.csv"
}

dfs = {}

for season, file in files.items():
    dfs[season] = pd.read_csv(file)
    dfs[season]["season"] = season

print("Ok")

Files loaded successfully.


Controllo dimensioni dataset

In [8]:
# Check number of matches per season

for season, df in dfs.items():
    print(season, ":", df.shape)

2018-19 : (380, 62)
2019-20 : (380, 106)
2020-21 : (380, 106)
2021-22 : (380, 106)
2022-23 : (380, 106)


In [9]:
dfs["2018-19"].head()

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSCH,PSCD,PSCA,season
0,I1,18/08/2018,Chievo,Juventus,2,3,A,1,1,D,...,19,2.00,1.68,1.64,2.38,2.29,18.84,6.42,1.22,2018-19
1,I1,18/08/2018,Lazio,Napoli,1,2,A,1,1,D,...,20,0.00,2.12,2.07,1.83,1.79,2.78,3.57,2.59,2018-19
2,I1,19/08/2018,Bologna,Spal,0,1,A,0,0,D,...,19,-0.25,1.97,1.92,1.99,1.94,2.31,3.18,3.59,2018-19
3,I1,19/08/2018,Empoli,Cagliari,2,0,H,1,0,H,...,19,-0.25,1.98,1.91,1.98,1.94,2.54,3.42,2.95,2018-19
4,I1,19/08/2018,Parma,Udinese,2,2,D,1,0,H,...,20,0.00,1.81,1.77,2.18,2.10,2.80,3.24,2.78,2018-19


In [11]:
dfs["2018-19"].info()


<class 'pandas.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 62 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Div       380 non-null    str    
 1   Date      380 non-null    str    
 2   HomeTeam  380 non-null    str    
 3   AwayTeam  380 non-null    str    
 4   FTHG      380 non-null    int64  
 5   FTAG      380 non-null    int64  
 6   FTR       380 non-null    str    
 7   HTHG      380 non-null    int64  
 8   HTAG      380 non-null    int64  
 9   HTR       380 non-null    str    
 10  HS        380 non-null    int64  
 11  AS        380 non-null    int64  
 12  HST       380 non-null    int64  
 13  AST       380 non-null    int64  
 14  HF        380 non-null    int64  
 15  AF        380 non-null    int64  
 16  HC        380 non-null    int64  
 17  AC        380 non-null    int64  
 18  HY        380 non-null    int64  
 19  AY        380 non-null    int64  
 20  HR        380 non-null    int64  
 21  AR  

In [12]:
# Check whether all CSV files have the same columns

base_cols = list(dfs["2018-19"].columns)

for season, df in dfs.items():
    same_columns = list(df.columns) == base_cols
    print(season, "same columns as 2018-19:", same_columns)

2018-19 same columns as 2018-19: True
2019-20 same columns as 2018-19: False
2020-21 same columns as 2018-19: False
2021-22 same columns as 2018-19: False
2022-23 same columns as 2018-19: False


In [13]:
# Essential columns for our project

essential_cols = [
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
    "HY",
    "AY"
]

for season, df in dfs.items():
    print("\nSeason:", season)
    for col in essential_cols:
        print(col, ":", col in df.columns)


Season: 2018-19
Date : True
HomeTeam : True
AwayTeam : True
FTHG : True
FTAG : True
FTR : True
HY : True
AY : True

Season: 2019-20
Date : True
HomeTeam : True
AwayTeam : True
FTHG : True
FTAG : True
FTR : True
HY : True
AY : True

Season: 2020-21
Date : True
HomeTeam : True
AwayTeam : True
FTHG : True
FTAG : True
FTR : True
HY : True
AY : True

Season: 2021-22
Date : True
HomeTeam : True
AwayTeam : True
FTHG : True
FTAG : True
FTR : True
HY : True
AY : True

Season: 2022-23
Date : True
HomeTeam : True
AwayTeam : True
FTHG : True
FTAG : True
FTR : True
HY : True
AY : True


In [14]:
# Missing values in essential columns

for season, df in dfs.items():
    print("\nSeason:", season)
    print(df[essential_cols].isnull().sum())


Season: 2018-19
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HY          0
AY          0
dtype: int64

Season: 2019-20
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HY          0
AY          0
dtype: int64

Season: 2020-21
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HY          0
AY          0
dtype: int64

Season: 2021-22
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HY          0
AY          0
dtype: int64

Season: 2022-23
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HY          0
AY          0
dtype: int64


L'unico aspetta da correggere a questo punto risulta essere quello legato al controllo colonne tra stagioni

In [15]:
# Compare columns across seasons

base_cols = set(dfs["2018-19"].columns)

for season, df in dfs.items():
    cols = set(df.columns)
    
    missing_from_season = base_cols - cols
    extra_in_season = cols - base_cols
    
    print("\n" + "="*50)
    print("Season:", season)
    print("Missing compared to 2018-19:")
    print(sorted(missing_from_season))
    print("Extra compared to 2018-19:")
    print(sorted(extra_in_season))


Season: 2018-19
Missing compared to 2018-19:
[]
Extra compared to 2018-19:
[]

Season: 2019-20
Missing compared to 2018-19:
['Bb1X2', 'BbAH', 'BbAHh', 'BbAv<2.5', 'BbAv>2.5', 'BbAvA', 'BbAvAHA', 'BbAvAHH', 'BbAvD', 'BbAvH', 'BbMx<2.5', 'BbMx>2.5', 'BbMxA', 'BbMxAHA', 'BbMxAHH', 'BbMxD', 'BbMxH', 'BbOU']
Extra compared to 2018-19:
['AHCh', 'AHh', 'Avg<2.5', 'Avg>2.5', 'AvgA', 'AvgAHA', 'AvgAHH', 'AvgC<2.5', 'AvgC>2.5', 'AvgCA', 'AvgCAHA', 'AvgCAHH', 'AvgCD', 'AvgCH', 'AvgD', 'AvgH', 'B365<2.5', 'B365>2.5', 'B365AHA', 'B365AHH', 'B365C<2.5', 'B365C>2.5', 'B365CA', 'B365CAHA', 'B365CAHH', 'B365CD', 'B365CH', 'BWCA', 'BWCD', 'BWCH', 'IWCA', 'IWCD', 'IWCH', 'Max<2.5', 'Max>2.5', 'MaxA', 'MaxAHA', 'MaxAHH', 'MaxC<2.5', 'MaxC>2.5', 'MaxCA', 'MaxCAHA', 'MaxCAHH', 'MaxCD', 'MaxCH', 'MaxD', 'MaxH', 'P<2.5', 'P>2.5', 'PAHA', 'PAHH', 'PC<2.5', 'PC>2.5', 'PCAHA', 'PCAHH', 'Time', 'VCCA', 'VCCD', 'VCCH', 'WHCA', 'WHCD', 'WHCH']

Season: 2020-21
Missing compared to 2018-19:
['Bb1X2', 'BbAH', 'BbAHh'

In [16]:
# Check essential columns only

essential_cols = [
    "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR",
    "HY", "AY"
]

for season, df in dfs.items():
    missing_essential = [col for col in essential_cols if col not in df.columns]
    print(season, "missing essential columns:", missing_essential)

2018-19 missing essential columns: []
2019-20 missing essential columns: []
2020-21 missing essential columns: []
2021-22 missing essential columns: []
2022-23 missing essential columns: []


Lasciamo solo ciò che ci interessa e unifichiamo le stagioni

In [17]:
# Keep only essential columns and combine seasons

df_matches_list = []

for season, df in dfs.items():
    temp = df[essential_cols].copy()
    temp["season"] = season
    df_matches_list.append(temp)

df_matches = pd.concat(df_matches_list, ignore_index=True)

df_matches.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HY,AY,season
0,18/08/2018,Chievo,Juventus,2,3,A,2,0,2018-19
1,18/08/2018,Lazio,Napoli,1,2,A,0,0,2018-19
2,19/08/2018,Bologna,Spal,0,1,A,4,2,2018-19
3,19/08/2018,Empoli,Cagliari,2,0,H,3,3,2018-19
4,19/08/2018,Parma,Udinese,2,2,D,2,2,2018-19


In [18]:
df_matches.shape

(1900, 9)

In [19]:
df_matches.isnull().sum()

Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HY          0
AY          0
season      0
dtype: int64

In [21]:
df_matches["Date"] = pd.to_datetime(df_matches["Date"], dayfirst=True)

df_matches["Date"].min(), df_matches["Date"].max()

(Timestamp('2018-08-18 00:00:00'), Timestamp('2023-06-04 00:00:00'))

In [ ]:
# Number of matches by season
# ci aspettiamo 380 partite per stagione (10 partite ogni giornata, 38 giornate)

df_matches.groupby("season").size()

season
2018-19    380
2019-20    380
2020-21    380
2021-22    380
2022-23    380
dtype: int64

Check Finale result values

In [23]:
# Check final result values

df_matches["FTR"].value_counts()

FTR
H    788
A    624
D    488
Name: count, dtype: int64

In [24]:
df_matches["FTR"].unique()

<StringArray>
['A', 'H', 'D']
Length: 3, dtype: str

Controllo Squadre per stagione

In [ ]:
# Number of unique teams per season (20 teams )

for season, df in df_matches.groupby("season"):
    teams = set(df["HomeTeam"]).union(set(df["AwayTeam"]))
    print(season, ":", len(teams), "teams")

2018-19 : 20 teams
2019-20 : 20 teams
2020-21 : 20 teams
2021-22 : 20 teams
2022-23 : 20 teams


Creazione Match-id

In [26]:
# Create match_id

df_matches = df_matches.sort_values(["Date", "HomeTeam", "AwayTeam"]).reset_index(drop=True)
df_matches["match_id"] = np.arange(1, len(df_matches) + 1)

df_matches.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HY,AY,season,match_id
0,2018-08-18,Chievo,Juventus,2,3,A,2,0,2018-19,1
1,2018-08-18,Lazio,Napoli,1,2,A,0,0,2018-19,2
2,2018-08-19,Bologna,Spal,0,1,A,4,2,2018-19,3
3,2018-08-19,Empoli,Cagliari,2,0,H,3,3,2018-19,4
4,2018-08-19,Parma,Udinese,2,2,D,2,2,2018-19,5


Creazione variabile `no_fans`

In [27]:
# Define behind-closed-doors period

start_no_fans = pd.Timestamp("2020-06-20")
end_no_fans = pd.Timestamp("2021-05-23")

df_matches["no_fans"] = (
    (df_matches["Date"] >= start_no_fans) &
    (df_matches["Date"] <= end_no_fans)
).astype(int)

df_matches["no_fans"].value_counts()

no_fans
0    1396
1     504
Name: count, dtype: int64

In [28]:
# Number of no-fans matches by season

df_matches.groupby("season")["no_fans"].sum()

season
2018-19      0
2019-20    124
2020-21    380
2021-22      0
2022-23      0
Name: no_fans, dtype: int64

*Costruzione del panel team-match*

In [29]:
# Build team-match panel dataset

home_rows = pd.DataFrame({
    "match_id": df_matches["match_id"],
    "Date": df_matches["Date"],
    "season": df_matches["season"],
    "team": df_matches["HomeTeam"],
    "opponent": df_matches["AwayTeam"],
    "home": 1,
    "no_fans": df_matches["no_fans"],
    "goals_scored": df_matches["FTHG"],
    "goals_conceded": df_matches["FTAG"],
    "goal_diff": df_matches["FTHG"] - df_matches["FTAG"],
    "win": (df_matches["FTR"] == "H").astype(int),
    "yellows": df_matches["HY"],
    "opponent_yellows": df_matches["AY"],
    "yellow_diff": df_matches["HY"] - df_matches["AY"]
})

away_rows = pd.DataFrame({
    "match_id": df_matches["match_id"],
    "Date": df_matches["Date"],
    "season": df_matches["season"],
    "team": df_matches["AwayTeam"],
    "opponent": df_matches["HomeTeam"],
    "home": 0,
    "no_fans": df_matches["no_fans"],
    "goals_scored": df_matches["FTAG"],
    "goals_conceded": df_matches["FTHG"],
    "goal_diff": df_matches["FTAG"] - df_matches["FTHG"],
    "win": (df_matches["FTR"] == "A").astype(int),
    "yellows": df_matches["AY"],
    "opponent_yellows": df_matches["HY"],
    "yellow_diff": df_matches["AY"] - df_matches["HY"]
})

df_panel = pd.concat([home_rows, away_rows], ignore_index=True)

df_panel = df_panel.sort_values(["Date", "match_id", "home"], ascending=[True, True, False]).reset_index(drop=True)

df_panel.head()

,match_id,Date,season,team,opponent,home,no_fans,goals_scored,goals_conceded,goal_diff,win,yellows,opponent_yellows,yellow_diff
0,1,2018-08-18,2018-19,Chievo,Juventus,1,0,2,3,-1,0,2,0,2
1,1,2018-08-18,2018-19,Juventus,Chievo,0,0,3,2,1,1,0,2,-2
2,2,2018-08-18,2018-19,Lazio,Napoli,1,0,1,2,-1,0,0,0,0
3,2,2018-08-18,2018-19,Napoli,Lazio,0,0,2,1,1,1,0,0,0
4,3,2018-08-19,2018-19,Bologna,Spal,1,0,0,1,-1,0,4,2,2


Controllo dimensione del panel

In [30]:
# Check panel size

print("Match-level observations:", len(df_matches))
print("Team-match observations:", len(df_panel))
print("Expected team-match observations:", len(df_matches) * 2)

Match-level observations: 1900
Team-match observations: 3800
Expected team-match observations: 3800


In [31]:
# Check that each match appears exactly twice

match_counts = df_panel.groupby("match_id").size()

print(match_counts.value_counts())

2    1900
Name: count, dtype: int64


In [32]:
# Create stable team_id

team_mapping = {team: i + 1 for i, team in enumerate(sorted(df_panel["team"].unique()))}

df_panel["team_id"] = df_panel["team"].map(team_mapping)

df_panel[["team", "team_id"]].drop_duplicates().sort_values("team_id").head(20)

,team,team_id
14,Atalanta,1
1539,Benevento,2
4,Bologna,3
765,Brescia,4
7,Cagliari,5
0,Chievo,6
3049,Cremonese,7
1525,Crotone,8
6,Empoli,9
22,Fiorentina,10


Creazione team_id stabile

In [33]:
# Create stable team_id

team_mapping = {team: i + 1 for i, team in enumerate(sorted(df_panel["team"].unique()))}

df_panel["team_id"] = df_panel["team"].map(team_mapping)

df_panel[["team", "team_id"]].drop_duplicates().sort_values("team_id").head(20)

,team,team_id
14,Atalanta,1
1539,Benevento,2
4,Bologna,3
765,Brescia,4
7,Cagliari,5
0,Chievo,6
3049,Cremonese,7
1525,Crotone,8
6,Empoli,9
22,Fiorentina,10


In [34]:
# Key interaction term

df_panel["home_x_nofans"] = df_panel["home"] * df_panel["no_fans"]

df_panel[["home", "no_fans", "home_x_nofans"]].value_counts().sort_index()

home  no_fans  home_x_nofans
0     0        0                1396
      1        0                 504
1     0        0                1396
      1        1                 504
Name: count, dtype: int64

In [35]:
# Missing values in final panel variables

model_cols = [
    "team_id",
    "match_id",
    "Date",
    "season",
    "team",
    "opponent",
    "home",
    "no_fans",
    "home_x_nofans",
    "goals_scored",
    "goals_conceded",
    "goal_diff",
    "win",
    "yellows",
    "opponent_yellows",
    "yellow_diff"
]

df_panel[model_cols].isnull().sum()

team_id             0
match_id            0
Date                0
season              0
team                0
opponent            0
home                0
no_fans             0
home_x_nofans       0
goals_scored        0
goals_conceded      0
goal_diff           0
win                 0
yellows             0
opponent_yellows    0
yellow_diff         0
dtype: int64

Statistiche descrittive

In [36]:
# Descriptive statistics by home status and fan presence

desc = df_panel.groupby(["home", "no_fans"]).agg(
    N=("win", "count"),
    win_rate=("win", "mean"),
    avg_goals=("goals_scored", "mean"),
    avg_goal_diff=("goal_diff", "mean"),
    avg_yellows=("yellows", "mean"),
    avg_yellow_diff=("yellow_diff", "mean")
).round(3)

desc

N  win_rate  avg_goals  avg_goal_diff  avg_yellows  \
home no_fans                                                          
0    0        1396     0.325      1.262         -0.218        2.586   
     1         504     0.337      1.444         -0.222        2.173   
1    0        1396     0.414      1.480          0.218        2.198   
     1         504     0.417      1.667          0.222        2.183   

              avg_yellow_diff  
home no_fans                   
0    0                  0.388  
     1                 -0.010  
1    0                 -0.388  
     1                  0.010

Home advantage descrittivo

In [37]:
# Descriptive statistics by home status and fan presence

desc = df_panel.groupby(["home", "no_fans"]).agg(
    N=("win", "count"),
    win_rate=("win", "mean"),
    avg_goals=("goals_scored", "mean"),
    avg_goal_diff=("goal_diff", "mean"),
    avg_yellows=("yellows", "mean"),
    avg_yellow_diff=("yellow_diff", "mean")
).round(3)

desc

N  win_rate  avg_goals  avg_goal_diff  avg_yellows  \
home no_fans                                                          
0    0        1396     0.325      1.262         -0.218        2.586   
     1         504     0.337      1.444         -0.222        2.173   
1    0        1396     0.414      1.480          0.218        2.198   
     1         504     0.417      1.667          0.222        2.183   

              avg_yellow_diff  
home no_fans                   
0    0                  0.388  
     1                 -0.010  
1    0                 -0.388  
     1                  0.010

In [39]:
# Raw home advantage with and without fans

home_adv = df_panel.groupby(["no_fans", "home"]).agg(
    win_rate=("win", "mean"),
    avg_goal_diff=("goal_diff", "mean"),
    avg_yellow_diff=("yellow_diff", "mean")
).reset_index()

home_adv

,no_fans,home,win_rate,avg_goal_diff,avg_yellow_diff
0,0,0,0.325215,-0.217765,0.388252
1,0,1,0.414040,0.217765,-0.388252
2,1,0,0.337302,-0.222222,-0.009921
3,1,1,0.416667,0.222222,0.009921


In [40]:
# Descriptive Difference-in-Differences

with_fans = home_adv[home_adv["no_fans"] == 0]
no_fans = home_adv[home_adv["no_fans"] == 1]

ha_win_with_fans = (
    with_fans[with_fans["home"] == 1]["win_rate"].values[0]
    - with_fans[with_fans["home"] == 0]["win_rate"].values[0]
)

ha_win_no_fans = (
    no_fans[no_fans["home"] == 1]["win_rate"].values[0]
    - no_fans[no_fans["home"] == 0]["win_rate"].values[0]
)

did_win = ha_win_no_fans - ha_win_with_fans

ha_gd_with_fans = (
    with_fans[with_fans["home"] == 1]["avg_goal_diff"].values[0]
    - with_fans[with_fans["home"] == 0]["avg_goal_diff"].values[0]
)

ha_gd_no_fans = (
    no_fans[no_fans["home"] == 1]["avg_goal_diff"].values[0]
    - no_fans[no_fans["home"] == 0]["avg_goal_diff"].values[0]
)

did_gd = ha_gd_no_fans - ha_gd_with_fans

print("Home advantage - win rate")
print(f"With fans: {ha_win_with_fans:.3f}")
print(f"No fans:   {ha_win_no_fans:.3f}")
print(f"Raw DiD:   {did_win:.3f}")

print("\nHome advantage - goal difference")
print(f"With fans: {ha_gd_with_fans:.3f}")
print(f"No fans:   {ha_gd_no_fans:.3f}")
print(f"Raw DiD:   {did_gd:.3f}")

Home advantage - win rate
With fans: 0.089
No fans:   0.079
Raw DiD:   -0.009

Home advantage - goal difference
With fans: 0.436
No fans:   0.444
Raw DiD:   0.009


In [41]:
# Home advantage by season

season_ha = df_panel.groupby(["season", "home"]).agg(
    win_rate=("win", "mean"),
    avg_goal_diff=("goal_diff", "mean"),
    avg_yellow_diff=("yellow_diff", "mean")
).reset_index()

season_ha

,season,home,win_rate,avg_goal_diff,avg_yellow_diff
0,2018-19,0,0.278947,-0.286842,0.376316
1,2018-19,1,0.436842,0.286842,-0.376316
2,2019-20,0,0.360526,-0.205263,0.215789
3,2019-20,1,0.415789,0.205263,-0.215789
4,2020-21,0,0.336842,-0.200000,0.023684
5,2020-21,1,0.407895,0.200000,-0.023684
6,2021-22,0,0.352632,-0.139474,0.457895
7,2021-22,1,0.389474,0.139474,-0.457895
8,2022-23,0,0.313158,-0.263158,0.339474
9,2022-23,1,0.423684,0.263158,-0.339474


In [42]:
# Compute home advantage by season

season_home = season_ha[season_ha["home"] == 1].set_index("season")
season_away = season_ha[season_ha["home"] == 0].set_index("season")

season_summary = pd.DataFrame({
    "HA_win_rate": season_home["win_rate"] - season_away["win_rate"],
    "HA_goal_diff": season_home["avg_goal_diff"] - season_away["avg_goal_diff"],
    "HA_yellow_diff": season_home["avg_yellow_diff"] - season_away["avg_yellow_diff"]
}).round(3)

season_summary

,HA_win_rate,HA_goal_diff,HA_yellow_diff
season,,,
2018-19,0.158,0.574,-0.753
2019-20,0.055,0.411,-0.432
2020-21,0.071,0.400,-0.047
2021-22,0.037,0.279,-0.916
2022-23,0.111,0.526,-0.679


Creazione season dummies

In [44]:
# Create season dummies

df_panel["season"] = pd.Categorical(
    df_panel["season"],
    categories=["2018-19", "2019-20", "2020-21", "2021-22", "2022-23"],
    ordered=True
)

season_dummies = pd.get_dummies(df_panel["season"], prefix="season", drop_first=True)

df_panel = pd.concat([df_panel, season_dummies], axis=1)

season_dummies.head()

,season_2019-20,season_2020-21,season_2021-22,season_2022-23
0,False,False,False,False
1,False,False,False,False
2,False,False,False,False
3,False,False,False,False
4,False,False,False,False


In [45]:
season_cols = list(season_dummies.columns)
season_cols

['season_2019-20', 'season_2020-21', 'season_2021-22', 'season_2022-23']

In [46]:
# Set panel index

df_model = df_panel.set_index(["team_id", "match_id"])

df_model.head()

,,Date,season,team,opponent,home,no_fans,goals_scored,goals_conceded,goal_diff,win,...,yellow_diff,home_x_nofans,season_2019-20,season_2020-21,season_2021-22,season_2022-23,season_2019-20,season_2020-21,season_2021-22,season_2022-23
team_id,match_id,,,,,,,,,,,,,,,,,,,,,
6,1,2018-08-18,2018-19,Chievo,Juventus,1,0,2,3,-1,0,...,2,0,False,False,False,False,False,False,False,False
14,1,2018-08-18,2018-19,Juventus,Chievo,0,0,3,2,1,1,...,-2,0,False,False,False,False,False,False,False,False
15,2,2018-08-18,2018-19,Lazio,Napoli,1,0,1,2,-1,0,...,0,0,False,False,False,False,False,False,False,False
19,2,2018-08-18,2018-19,Napoli,Lazio,0,0,2,1,1,1,...,0,0,False,False,False,False,False,False,False,False
3,3,2018-08-19,2018-19,Bologna,Spal,1,0,0,1,-1,0,...,2,0,False,False,False,False,False,False,False,False


In [47]:
# Define regressors

main_regressors = [
    "home",
    "no_fans",
    "home_x_nofans"
] + season_cols

main_regressors

['home',
 'no_fans',
 'home_x_nofans',
 'season_2019-20',
 'season_2020-21',
 'season_2021-22',
 'season_2022-23']

In [48]:
# Main Fixed Effects model: outcome = goal_diff

model_gd = PanelOLS(
    df_model["goal_diff"],
    df_model[main_regressors],
    entity_effects=True
)

res_gd = model_gd.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(res_gd)

AttributeError: 'DataFrame' object has no attribute 'dtype'

In [51]:
df_panel.columns[df_panel.columns.duplicated()]

Index([], dtype='str')

In [50]:
# Remove duplicated columns, if any

df_panel = df_panel.loc[:, ~df_panel.columns.duplicated()].copy()

In [52]:
# Remove old season dummy columns if already present

df_panel = df_panel.drop(
    columns=[col for col in df_panel.columns if col.startswith("season_")],
    errors="ignore"
)

In [53]:
# Create season dummies

df_panel["season"] = pd.Categorical(
    df_panel["season"],
    categories=["2018-19", "2019-20", "2020-21", "2021-22", "2022-23"],
    ordered=True
)

season_dummies = pd.get_dummies(
    df_panel["season"],
    prefix="season",
    drop_first=True
).astype(int)

df_panel = pd.concat([df_panel, season_dummies], axis=1)

season_cols = list(season_dummies.columns)

season_cols

['season_2019-20', 'season_2020-21', 'season_2021-22', 'season_2022-23']

In [54]:
df_model = df_panel.set_index(["team_id", "match_id"])

In [55]:
main_regressors = [
    "home",
    "no_fans",
    "home_x_nofans"
] + season_cols

In [56]:
model_gd = PanelOLS(
    df_model["goal_diff"],
    df_model[main_regressors],
    entity_effects=True
)

res_gd = model_gd.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(res_gd)

                          PanelOLS Estimation Summary                           
Dep. Variable:              goal_diff   R-squared:                        0.0177
Estimator:                   PanelOLS   R-squared (Between):             -0.2376
No. Observations:                3800   R-squared (Within):               0.0177
Date:                Tue, May 12 2026   R-squared (Overall):              0.0058
Time:                        16:13:49   Log-likelihood                   -7288.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      9.6598
Entities:                          30   P-value                           0.0000
Avg Obs:                       126.67   Distribution:                  F(7,3763)
Min Obs:                       38.000                                           
Max Obs:                       190.00   F-statistic (robust):             13.606
                            

First Robustness Check

In [57]:
# Robustness check: outcome = win

model_win = PanelOLS(
    df_model["win"],
    df_model[main_regressors],
    entity_effects=True
)

res_win = model_win.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(res_win)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    win   R-squared:                        0.0094
Estimator:                   PanelOLS   R-squared (Between):              0.2215
No. Observations:                3800   R-squared (Within):               0.0094
Date:                Tue, May 12 2026   R-squared (Overall):              0.0945
Time:                        16:16:08   Log-likelihood                   -2393.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      5.0979
Entities:                          30   P-value                           0.0000
Avg Obs:                       126.67   Distribution:                  F(7,3763)
Min Obs:                       38.000                                           
Max Obs:                       190.00   F-statistic (robust):             5.3208
                            

Stimiamo il modello arbitrale

In [58]:
# Referee bias mechanism: outcome = yellow_diff

model_yellow = PanelOLS(
    df_model["yellow_diff"],
    df_model[main_regressors],
    entity_effects=True
)

res_yellow = model_yellow.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(res_yellow)

                          PanelOLS Estimation Summary                           
Dep. Variable:            yellow_diff   R-squared:                        0.0350
Estimator:                   PanelOLS   R-squared (Between):             -1.1876
No. Observations:                3800   R-squared (Within):               0.0350
Date:                Tue, May 12 2026   R-squared (Overall):             -0.0029
Time:                        17:09:53   Log-likelihood                   -7531.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      19.505
Entities:                          30   P-value                           0.0000
Avg Obs:                       126.67   Distribution:                  F(7,3763)
Min Obs:                       38.000                                           
Max Obs:                       190.00   F-statistic (robust):             25.098
                            